### Polars

In [1]:
import polars as pl

df = pl.DataFrame({
    "nom": ["Alice", "Bob", "Claire"],
    "age": [28, 35, 22],
    "ville": ["Paris", "Lyon", "Paris"]
})

print(df)

shape: (3, 3)
┌────────┬─────┬───────┐
│ nom    ┆ age ┆ ville │
│ ---    ┆ --- ┆ ---   │
│ str    ┆ i64 ┆ str   │
╞════════╪═════╪═══════╡
│ Alice  ┆ 28  ┆ Paris │
│ Bob    ┆ 35  ┆ Lyon  │
│ Claire ┆ 22  ┆ Paris │
└────────┴─────┴───────┘


In [2]:
df.show()

nom,age,ville
str,i64,str
"""Alice""",28,"""Paris"""
"""Bob""",35,"""Lyon"""
"""Claire""",22,"""Paris"""


In [3]:
# Sélection de colonnes
df.select(["nom", "age"])

nom,age
str,i64
"""Alice""",28
"""Bob""",35
"""Claire""",22


In [4]:

# Filtre
df.filter(pl.col("age") > 25)

nom,age,ville
str,i64,str
"""Alice""",28,"""Paris"""
"""Bob""",35,"""Lyon"""


In [5]:

# Ajout d'une colonne
df.with_columns(
    (pl.col("age") + 1).alias("age_an_prochain")
)

nom,age,ville,age_an_prochain
str,i64,str,i64
"""Alice""",28,"""Paris""",29
"""Bob""",35,"""Lyon""",36
"""Claire""",22,"""Paris""",23


In [6]:
(
    df
    .filter(pl.col("ville") == "Paris")
    .group_by("ville")
    .agg(
        pl.col("age").mean().alias("age_moyen")
    )
)

ville,age_moyen
str,f64
"""Paris""",25.0


In [18]:
# Lecture des colonnes d'un fichier CSV en mode "lazy"
pl.scan_csv("../00-data/individus.csv").columns

/var/folders/2g/xf2xxxf55y3fdxbw2x_rss480000gn/T/ipykernel_24173/968581662.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  pl.scan_csv("../00-data/individus.csv").columns


['id_individu',
 'id_menage',
 'region',
 'departement',
 'milieu_residence',
 'nom',
 'prenom',
 'sexe',
 'age',
 'date_naissance',
 'lien_chef_menage',
 'situation_matrimoniale',
 'niveau_instruction',
 'sait_lire_ecrire',
 'situation_activite',
 'secteur_activite',
 'nationalite',
 'type_logement',
 'nb_pieces',
 'source_eau',
 'electricite']

In [ ]:
# scan_csv permet de lire un fichier CSV en mode "lazy", c'est-à-dire que les opérations sont planifiées mais pas encore exécutées.
result = (
    pl.scan_csv("../00-data/regions.csv")
      .filter(pl.col("superficie_km2") > 500)
      .with_columns(
          (pl.when(pl.col("part_urbaine") > 0.5).then(1).otherwise(0)).alias("urb_class")
      )
      .group_by("urb_class")
      .agg(pl.sum("poids_demographique"))
)

In [12]:
result.collect()

urb_class,poids_demographique
i32,f64
1,0.273
0,0.727


In [19]:
# Execution en mode streaming (exécution en continu)
import polars as pl

result = (
    pl.scan_csv("../00-data/individus.csv")
      .filter(pl.col("sexe") == "M")
      .group_by("age")
      .agg(pl.len().alias("effectif_age_homme"))
      .collect(engine="streaming")
)

In [20]:
result

age,effectif_age_homme
i64,u32
34,41598
58,18040
46,30777
55,21645
85,984
…,…
79,2180
53,24168
0,11842


In [21]:
# SQL
df = pl.DataFrame(
    {
        "country": ["USA", "USA", "USA", "USA", "USA", "Netherlands"],
        "city": [
            "New York",
            "Los Angeles",
            "Chicago",
            "Houston",
            "Phoenix",
            "Amsterdam",
        ],
        "population": [8399000, 3997000, 2705000, 2320000, 1680000, 900000],
    }
)

ctx = pl.SQLContext(population=df, eager=True)

print(ctx.execute("SELECT * FROM population"))

shape: (6, 3)
┌─────────────┬─────────────┬────────────┐
│ country     ┆ city        ┆ population │
│ ---         ┆ ---         ┆ ---        │
│ str         ┆ str         ┆ i64        │
╞═════════════╪═════════════╪════════════╡
│ USA         ┆ New York    ┆ 8399000    │
│ USA         ┆ Los Angeles ┆ 3997000    │
│ USA         ┆ Chicago     ┆ 2705000    │
│ USA         ┆ Houston     ┆ 2320000    │
│ USA         ┆ Phoenix     ┆ 1680000    │
│ Netherlands ┆ Amsterdam   ┆ 900000     │
└─────────────┴─────────────┴────────────┘


------

### DuckDB

In [28]:
import duckdb
duckdb.sql("SELECT 42").show()

┌───────┐
│  42   │
│ int32 │
├───────┤
│    42 │
└───────┘



In [ ]:

con = duckdb.connect()

df = con.sql("""
SELECT *
FROM '../00-data/entreprises.csv'
""").df()

print(df.head())

  id_entreprise                     raison_sociale forme_juridique  \
0    ENT0000001  Atlantique Industries Association     Association   
1    ENT0000002              Saloum Services SUARL           SUARL   
2    ENT0000003               Baobab Conseil SUARL           SUARL   
3    ENT0000004               Teranga Services GIE             GIE   
4    ENT0000005              Ferlo Industries SARL            SARL   

              secteur_activite       region departement  annee_creation  \
0      Administration publique     Diourbel    Diourbel            1994   
1                        Santé  Saint-Louis       Podor            1993   
2                          BTP        Kolda   Vélingara            1991   
3                     Commerce     Kédougou    Salémata            2003   
4  Agriculture, élevage, pêche   Ziguinchor    Oussouye            2010   

   effectif  chiffre_affaires_fcfa statut  
0        10            127587561.0  Actif  
1         3             47861087.0  Acti

In [ ]:
# Lire un fichier CSV read_csv() qui permet de lire un fichier CSV et de le traiter comme une table relationnelle.
reg = con.sql("""
SELECT *
FROM read_csv('../00-data/regions.csv')
""").df()

reg.head()

,code_region,region,chef_lieu,superficie_km2,poids_demographique,part_urbaine
0,DK,Dakar,Dakar,547,0.230,0.97
1,TH,Thiès,Thiès,6601,0.130,0.48
2,DB,Diourbel,Diourbel,4824,0.110,0.40
3,KL,Kaolack,Kaolack,5357,0.072,0.38
4,SL,Saint-Louis,Saint-Louis,19034,0.065,0.44


In [27]:
# Depuis un dataframe Polas
polars_df = pl.DataFrame({"a": [42]})
duckdb.sql("SELECT * FROM polars_df")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┐
│   a   │
│ int64 │
├───────┤
│    42 │
└───────┘

In [29]:
# Lecture de pandas DataFrame
import pandas as pd

pandas_df = pd.DataFrame({"a": [42]})
duckdb.sql("SELECT * FROM pandas_df")

┌───────┐
│   a   │
│ int64 │
├───────┤
│    42 │
└───────┘

In [30]:
# Créer un table à la volée

# create a connection to a file called 'file.db'
con = duckdb.connect("../00-data/file.db")
# create a table and load data into it
con.sql("CREATE TABLE test (i INTEGER)")
con.sql("INSERT INTO test VALUES (42)")
# query the table
con.table("test").show()
# explicitly close the connection
con.close()
# Note: connections also closed implicitly when they go out of scope

┌───────┐
│   i   │
│ int32 │
├───────┤
│    42 │
└───────┘

